## 准备数据

In [6]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [7]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [8]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(
            tf.random.normal([28*28, 100], stddev=0.1)
        )

        self.b1 = tf.Variable(
            tf.zeros([100])
        )

        self.W2 = tf.Variable(
            tf.random.normal([100, 10], stddev=0.1)
        )

        self.b2 = tf.Variable(
            tf.zeros([10])
        )
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, [-1, 28*28])

        h1 = tf.matmul(x, self.W1) + self.b1
        h1 = tf.nn.relu(h1)

        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [9]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.1*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [10]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.4382932 ; accuracy 0.11643333
epoch 1 : loss 2.2785685 ; accuracy 0.14388333
epoch 2 : loss 2.1812556 ; accuracy 0.19225
epoch 3 : loss 2.1016102 ; accuracy 0.2499
epoch 4 : loss 2.0300379 ; accuracy 0.30768332
epoch 5 : loss 1.9633155 ; accuracy 0.36
epoch 6 : loss 1.9001503 ; accuracy 0.4093
epoch 7 : loss 1.8399174 ; accuracy 0.45516667
epoch 8 : loss 1.7822753 ; accuracy 0.49615
epoch 9 : loss 1.7270112 ; accuracy 0.5316
epoch 10 : loss 1.6739982 ; accuracy 0.56345
epoch 11 : loss 1.6231211 ; accuracy 0.5885
epoch 12 : loss 1.5742805 ; accuracy 0.61198336
epoch 13 : loss 1.5274088 ; accuracy 0.63053334
epoch 14 : loss 1.4824699 ; accuracy 0.6475
epoch 15 : loss 1.4394418 ; accuracy 0.6620833
epoch 16 : loss 1.3982546 ; accuracy 0.6742
epoch 17 : loss 1.3588709 ; accuracy 0.6857333
epoch 18 : loss 1.3212613 ; accuracy 0.69605
epoch 19 : loss 1.2853857 ; accuracy 0.70525
epoch 20 : loss 1.2511986 ; accuracy 0.7132
epoch 21 : loss 1.2186476 ; accuracy 0.72065
epoch 22